# Audit of MIMA Imputation
## MIMA applied to the study's hourly base and compared with hybrid imputation

MIMA is the deep-learning imputation approach developed for SIMA by Gómez, Sánchez, Medina, Montibeller and Pérez. Its architecture includes SAITS and a CSDI stage; this notebook checks their effective contribution. Training on the hourly base follows the supplied protocol: 168-hour windows, 50 epochs and a 20% artificial mask for auditing. Training takes time on a CPU. Materials are in `../data_processed/mima/`: `mima_pipeline.py`, trained model `mima_pato_model.pt` and audit `mima_auditoria.json`. This notebook loads and examines their results.

**Data access.** The SIMA measurements used here are not
redistributed in this repository (SIMA provided them for this study on the condition
that they are not disseminated); the repository holds the code, the official school
calendar and all derived results. This notebook reads files that notebooks 00 and 01
build from those measurements, and its configuration cell stops with a clear message
if they are missing. To rebuild them, request the annual workbooks from SIMA, place
them in `data_raw/` and run notebook 00 (its header gives the exact file format).

**Role in the analysis.** Sensitivity to missing data: audit of the alternative MIMA imputation and comparison with the hybrid one. The analytical hierarchy of the whole repository is stated in the README.

## 1. Initial setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']
colores_contaminantes = {'CO': '#3498db', 'NO2': '#e74c3c', 'O3': '#2ecc71',
                         'PM10': '#f39c12', 'PM2.5': '#9b59b6'}

output_dir = Path('../data_processed')
mima_dir = output_dir / 'mima'

print('Libraries imported successfully')

# Check required inputs (restricted data; see heading)
# Input-data check (restricted data; see heading)
archivos_requeridos = [output_dir / 'panel_pico_hibrido.csv',
                       output_dir / 'panel_pico_MIMA.csv',
                       output_dir / 'mima/mima_auditoria.json']
faltantes = [str(a) for a in archivos_requeridos if not a.exists()]
if faltantes:
    raise FileNotFoundError(
        ''
        ''
        "Files derived from the SIMA measurements are missing; they are not redistributed in this repository. "
        "Request the annual workbooks from SIMA, place them in data_raw/ and run notebooks 00 and 01. "
        f"Missing: {faltantes}")
print(f"Input data found: {len(archivos_requeridos)} files")

Libraries imported successfully
Input data found: 3 files


## 2. Artificial-gap audit metrics

The audit hides 20% of observed entries, imputes them and compares predictions against their known values. Metrics are reported with and without CSDI. This is an internal artificial-masking diagnostic, not validation on an independent monitoring period.

### Metrics

For deliberately hidden OBSERVED entries, compare known values $y_i$ with imputations $\hat y_i$:

$$ MAE=\tfrac1n\sum_i|y_i-\hat y_i|,\qquad RMSE=\sqrt{\tfrac1n\sum_i(y_i-\hat y_i)^2},\qquad R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}. $$

MAE weights absolute errors equally; RMSE penalizes large errors more strongly; $R^2$ compares squared prediction error with variation around the observed mean. Pooled metrics across pollutants with different units should not be interpreted as uniform accuracy for every pollutant.

In [2]:
auditoria = json.load(open(mima_dir / 'mima_auditoria.json'))

tabla_auditoria = pd.DataFrame([auditoria['saits'], auditoria['saits_csdi_ens10']])
tabla_auditoria = tabla_auditoria.set_index('tag')

print("=" * 70)
print('MIMA AUDIT WITH ARTIFICIALLY MASKED VALUES')
print("=" * 70)
print(tabla_auditoria.T)
print(f"\nNote: {auditoria['nota']}")

tabla_auditoria.T.to_csv(mima_dir / 'tabla_auditoria.csv')
print(f"\nSaved to: {mima_dir / 'tabla_auditoria.csv'}")

MIMA AUDIT WITH ARTIFICIALLY MASKED VALUES
tag                        SAITS  SAITS+CSDI(ens10)
n_celdas_evaluadas   104557.0000        104557.0000
MAE                       6.2467             6.2509
RMSE                     12.2094            12.2095
MRE_pct                  31.1600            31.1100
R2                        0.8072             0.8072
Pearson                   0.8988             0.8988
viol_fisica_PM_pct        0.4200             0.3900
quimica_NO2_O3_corr      -0.2737            -0.2737

Note: In the included audit, SAITS and the CSDI ensemble have very similar, but not identical, metrics; final analysis imputation uses deterministic SAITS.

Saved to: ../data_processed/mima/tabla_auditoria.csv


In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metricas = [('MAE', 'Mean absolute error'), ('RMSE', 'Root mean squared error'), ('R2', 'R²')]
for ax, (metrica, titulo) in zip(axes, metricas):
    valores = [auditoria['saits'][metrica], auditoria['saits_csdi_ens10'][metrica]]
    bars = ax.bar(['SAITS only', 'SAITS + CSDI'], valores,
                  color=['#3498db', '#e74c3c'], alpha=0.8, edgecolor='black')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_title(titulo, fontsize=12, fontweight='bold')

fig.suptitle('Nearly identical metrics with and without CSDI: the effective imputation is SAITS\n(internal masking diagnostic for the included trained model)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(mima_dir / '01_metricas_auditoria.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {mima_dir / '01_metricas_auditoria.png'}")

Plot saved to: ../data_processed/mima/01_metricas_auditoria.png


## 3. Physical and chemical checks

Two diagnostics complement the prediction-error metrics:

- Physical consistency: PM2.5 is a subset of PM10, so entries where measured PM2.5 exceeds PM10 deserve examination for measurement or imputation inconsistencies.
- Chemical co-movement: examine the NO2–O3 correlation. Its sign is descriptive rather than a universal validity rule because chemistry, sunlight, transport and emissions jointly affect both pollutants.

In [4]:
print("=" * 70)
print('PHYSICAL AND CHEMICAL DIAGNOSTICS')
print("=" * 70)
print(f"Imputed cells with PM2.5 > PM10: {auditoria['saits']['viol_fisica_PM_pct']}%")
print(f"NO2-O3 correlation in the imputed dataset: {auditoria['saits']['quimica_NO2_O3_corr']}")
print('=> These descriptive checks alone do not establish physical or chemical validity.')

PHYSICAL AND CHEMICAL DIAGNOSTICS
Imputed cells with PM2.5 > PM10: 0.42%
NO2-O3 correlation in the imputed dataset: -0.2737
=> These descriptive checks alone do not establish physical or chemical validity.


## 4. Does the analysis depend on imputation? Hybrid versus MIMA

The main sensitivity check runs the analytical pipeline under both imputations. Here we compare their day–monitoring-unit aggregates.

In [5]:
panel_hibrido = pd.read_csv('../data_processed/panel_pico_hibrido.csv', parse_dates=['date'])
panel_mima = pd.read_csv('../data_processed/panel_pico_MIMA.csv', parse_dates=['date'])

emparejado = panel_hibrido.merge(panel_mima, on=['date', 'estacion'], suffixes=('_h', '_m'))

print("=" * 70)
print('CORRELATION BETWEEN AGGREGATES: HYBRID vs MIMA IMPUTATION')
print("=" * 70)
correlaciones = {}
for c in contaminantes:
    correlaciones[c] = emparejado[f'{c}_h'].corr(emparejado[f'{c}_m'])
    print(f"  {c}: r = {correlaciones[c]:.4f}")

pd.Series(correlaciones, name='correlacion').to_csv(mima_dir / 'correlaciones_hibrido_vs_mima.csv')
print(f"\nSaved to: {mima_dir / 'correlaciones_hibrido_vs_mima.csv'}")

CORRELATION BETWEEN AGGREGATES: HYBRID vs MIMA IMPUTATION
  CO: r = 0.9952
  NO2: r = 0.9981
  O3: r = 0.9708
  PM10: r = 0.9988
  PM2.5: r = 0.9882

Saved to: ../data_processed/mima/correlaciones_hibrido_vs_mima.csv


In [6]:
fig, axes = plt.subplots(1, 5, figsize=(17, 4))

for ax, c in zip(axes, contaminantes):
    ax.scatter(emparejado[f'{c}_h'], emparejado[f'{c}_m'], s=4, alpha=0.25,
               color=colores_contaminantes[c])
    limites = [emparejado[[f'{c}_h', f'{c}_m']].min().min(), emparejado[[f'{c}_h', f'{c}_m']].max().max()]
    ax.plot(limites, limites, color='black', linewidth=1, linestyle='--')
    ax.set_title(f'{c}  (r = {correlaciones[c]:.3f})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Hybrid', fontsize=10)
axes[0].set_ylabel('MIMA', fontsize=10, fontweight='bold')

fig.suptitle('Arrival-window day-unit aggregates: agreement between imputation methods',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(mima_dir / '02_hibrido_vs_mima_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to: {mima_dir / '02_hibrido_vs_mima_scatter.png'}")

print('\nCOMPLETED')

Plot saved to: ../data_processed/mima/02_hibrido_vs_mima_scatter.png

COMPLETED
